## This the Datapreprocessing notebook for the CNN model


THis file for data processing for both ML and DL models that i am using in this project

In [ ]:
import numpy as np
import lightkurve as lk
from scipy import interpolate
from scipy.signal import medfilt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


class KeplerPreprocessor:
    """
    Preprocessing pipeline for Kepler light curves following Shallue & Vanderburg (2018).
    
    Paper: "Identifying Exoplanets with Deep Learning"
    https://doi.org/10.3847/1538-3881/aa9e09
    """
    
    def __init__(self, period, t0, duration, mission='Kepler'):
        """
        Args:
            period: Orbital period in days
            t0: Time of first transit (BJD)
            duration: Transit duration in hours
            mission: 'Kepler' or 'TESS'
        """
        self.period = period
        self.t0 = t0
        self.duration = duration / 24.0  # Convert hours to days
        self.mission = mission
        
    def download_lightcurve(self, target_id, download_dir=None):
        """
        Download light curve from MAST.
        
        Args:
            target_id: KIC/TIC ID or target name
            download_dir: Optional directory to save files
            
        Returns:
            LightCurve object
        """
        print(f"Downloading {self.mission} data for {target_id}...")
        
        search_result = lk.search_lightcurve(target_id, mission=self.mission)
        
        if len(search_result) == 0:
            raise ValueError(f"No {self.mission} data found for {target_id}")
        
        # Download all quarters/sectors
        lc_collection = search_result.download_all(download_dir=download_dir)
        
        # Stitch together and use PDCSAP flux
        lc = lc_collection.stitch()
        
        return lc
    
    def remove_known_transits(self, lc, known_periods, known_t0s, known_durations):
        """
        Remove transits of known planets in the system.
        
        Args:
            lc: LightCurve object
            known_periods: List of periods (days)
            known_t0s: List of transit times (BJD)
            known_durations: List of durations (hours)
            
        Returns:
            LightCurve with known transits masked
        """
        mask = np.ones(len(lc.time), dtype=bool)
        
        for p, t0, dur in zip(known_periods, known_t0s, known_durations):
            dur_days = dur / 24.0
            # Calculate phase
            phase = (lc.time.value - t0) % p
            # Mask points within transit
            in_transit = (phase < dur_days) | (phase > (p - dur_days))
            mask &= ~in_transit
        
        return lc[mask]
    
    def flatten_lightcurve(self, lc, break_point_spacings=None):
        """
        Flatten light curve using basis spline with optimal break-point spacing (BIC).
        
        Args:
            lc: LightCurve object
            break_point_spacings: List of spacings to test (in days). 
                                  If None, uses default range.
        
        Returns:
            Flattened flux array, time array
        """
        if break_point_spacings is None:
            # Test range of break-point spacings (in days)
            break_point_spacings = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
        
        time = lc.time.value
        flux = lc.flux.value
        
        # Remove NaNs
        valid = np.isfinite(time) & np.isfinite(flux)
        time = time[valid]
        flux = flux[valid]
        
        # Identify in-transit points to exclude from spline fit
        phase = (time - self.t0) % self.period
        in_transit = (phase < self.duration) | (phase > (self.period - self.duration))
        
        best_bic = np.inf
        best_spline = None
        best_spacing = None
        
        for spacing in break_point_spacings:
            # Create knot sequence
            n_knots = int((time[-1] - time[0]) / spacing)
            if n_knots < 4:
                continue
                
            knots = np.linspace(time[0], time[-1], n_knots)
            
            # Iterative outlier rejection
            mask = ~in_transit.copy()
            spline = None
            
            for iteration in range(3):
                try:
                    # Fit spline to out-of-transit points
                    spline = interpolate.LSQUnivariateSpline(
                        time[mask], flux[mask], knots[1:-1], k=3
                    )
                    
                    # Compute residuals
                    model = spline(time[mask])
                    residuals = flux[mask] - model
                    
                    # Remove 3-sigma outliers
                    sigma = np.std(residuals)
                    outliers = np.abs(residuals) > 3 * sigma
                    mask[np.where(mask)[0][outliers]] = False
                    
                except Exception as e:
                    spline = None
                    break
            
            if spline is None:
                continue
            
            # Calculate BIC: BIC = n*ln(RSS/n) + k*ln(n)
            model = spline(time[mask])
            residuals = flux[mask] - model
            rss = np.sum(residuals**2)
            n = len(time[mask])
            k = len(knots)
            
            bic = n * np.log(rss / n) + k * np.log(n)
            
            if bic < best_bic:
                best_bic = bic
                best_spline = spline
                best_spacing = spacing
        
        if best_spline is None:
            raise ValueError("Failed to fit spline with any break-point spacing")
        
        print(f"Optimal break-point spacing: {best_spacing} days (BIC: {best_bic:.2f})")
        
        # Flatten by dividing by spline
        # Interpolate over in-transit points
        spline_model = best_spline(time)
        
        # For in-transit points, use linear interpolation of nearby spline values
        for i in np.where(in_transit)[0]:
            if i > 0 and i < len(time) - 1:
                # Find nearest out-of-transit points
                left_idx = i - 1
                right_idx = i + 1
                while left_idx >= 0 and in_transit[left_idx]:
                    left_idx -= 1
                while right_idx < len(time) and in_transit[right_idx]:
                    right_idx += 1
                
                if left_idx >= 0 and right_idx < len(time):
                    # Linear interpolation
                    t_left, t_right = time[left_idx], time[right_idx]
                    s_left, s_right = spline_model[left_idx], spline_model[right_idx]
                    spline_model[i] = s_left + (s_right - s_left) * (time[i] - t_left) / (t_right - t_left)
        
        flattened_flux = flux / spline_model
        
        return flattened_flux, time
    
    def fold_and_bin(self, time, flux, bin_width, bin_spacing, n_bins):
        """
        Fold light curve on period and bin.
        
        Args:
            time: Time array
            flux: Flux array
            bin_width: Width of each bin (δ)
            bin_spacing: Distance between bin centers (λ)
            n_bins: Number of bins
            
        Returns:
            Binned flux array
        """
        # Fold on period, centered on transit
        phase = ((time - self.t0) % self.period) / self.period
        phase[phase > 0.5] -= 1.0  # Center on transit at phase=0
        
        # Convert phase to time units
        phase_time = phase * self.period
        
        # Create bins
        half_span = (n_bins * bin_spacing) / 2.0
        bin_centers = np.linspace(-half_span, half_span, n_bins)
        
        binned_flux = np.zeros(n_bins)
        
        for i, center in enumerate(bin_centers):
            # Find points within bin
            in_bin = (phase_time >= center - bin_width/2) & (phase_time < center + bin_width/2)
            
            if np.sum(in_bin) > 0:
                # Use median (robust to outliers)
                binned_flux[i] = np.median(flux[in_bin])
            else:
                # Interpolate if no points in bin
                binned_flux[i] = np.nan
        
        # Fill NaNs with interpolation
        if np.any(np.isnan(binned_flux)):
            valid = ~np.isnan(binned_flux)
            if np.sum(valid) > 1:
                binned_flux = np.interp(
                    bin_centers, bin_centers[valid], binned_flux[valid]
                )
        
        return binned_flux
    
    def create_global_view(self, time, flux):
        """
        Create global view: 2001 bins covering entire folded light curve.
        
        Args:
            time: Time array
            flux: Flattened flux array
            
        Returns:
            Global view array (2001 points)
        """
        n_bins = 2001
        lambda_global = self.period / n_bins
        delta_global = lambda_global  # Non-overlapping
        
        return self.fold_and_bin(time, flux, delta_global, lambda_global, n_bins)
    
    def create_local_view(self, time, flux):
        """
        Create local view: 201 bins around transit (±4 transit durations).
        
        Args:
            time: Time array
            flux: Flattened flux array
            
        Returns:
            Local view array (201 points)
        """
        n_bins = 201
        k = 4  # Transit durations on each side
        lambda_local = (2 * k * self.duration) / n_bins
        delta_local = 0.16 * self.duration  # Overlapping bins
        
        return self.fold_and_bin(time, flux, delta_local, lambda_local, n_bins)
    
    def normalize(self, global_view, local_view):
        """
        Normalize views: median=0, min=-1
        
        Args:
            global_view: Global view array
            local_view: Local view array
            
        Returns:
            Normalized global and local views
        """
        def normalize_array(arr):
            arr = arr - np.median(arr)  # Median = 0
            min_val = np.min(arr)
            if min_val < 0:
                arr = arr / abs(min_val)  # Min = -1
            return arr
        
        return normalize_array(global_view), normalize_array(local_view)
    
    def preprocess(self, lc, known_periods=None, known_t0s=None, known_durations=None):
        """
        Full preprocessing pipeline.
        
        Args:
            lc: LightCurve object
            known_periods: List of known planet periods to remove (optional)
            known_t0s: List of known planet transit times (optional)
            known_durations: List of known planet durations (optional)
            
        Returns:
            global_view (2001), local_view (201)
        """
        # Step 1: Remove known transits if provided
        if known_periods is not None:
            print("Removing known planet transits...")
            lc = self.remove_known_transits(lc, known_periods, known_t0s, known_durations)
        
        # Step 2: Flatten light curve
        print("Flattening light curve...")
        flux, time = self.flatten_lightcurve(lc)
        
        # Step 3: Create views
        print("Creating global view (2001 bins)...")
        global_view = self.create_global_view(time, flux)
        
        print("Creating local view (201 bins)...")
        local_view = self.create_local_view(time, flux)
        
        # Step 4: Normalize
        print("Normalizing...")
        global_view, local_view = self.normalize(global_view, local_view)
        
        return global_view, local_view


# ============================================================
# USAGE EXAMPLE
# ============================================================

def process_single_target(target_id, period, t0, duration, 
                         known_periods=None, known_t0s=None, known_durations=None):
    """
    Process a single target and return preprocessed views.
    
    Args:
        target_id: KIC/TIC ID or target name
        period: Orbital period (days)
        t0: Time of first transit (BJD)
        duration: Transit duration (hours)
        known_periods: List of other planets' periods (optional)
        known_t0s: List of other planets' t0s (optional)
        known_durations: List of other planets' durations (optional)
    
    Returns:
        global_view (2001), local_view (201)
    """
    preprocessor = KeplerPreprocessor(period, t0, duration)
    
    # Download data
    lc = preprocessor.download_lightcurve(target_id)
    
    # Preprocess
    global_view, local_view = preprocessor.preprocess(
        lc, known_periods, known_t0s, known_durations
    )
    
    return global_view, local_view


def process_multiple_targets_to_csv(targets_df, output_dir='./'):
    """
    Process multiple targets and save to CSV files.
    
    Args:
        targets_df: DataFrame with columns:
                   ['target_id', 'period', 't0', 'duration', 'label']
                   Optional: 'known_periods', 'known_t0s', 'known_durations' 
                            (as comma-separated strings)
        output_dir: Directory to save CSV files
    
    Saves:
        all_global.csv: 2001 features + label
        all_local.csv: 201 features + label
    """
    global_views = []
    local_views = []
    labels = []
    
    for idx, row in targets_df.iterrows():
        print(f"\n{'='*60}")
        print(f"Processing {idx+1}/{len(targets_df)}: {row['target_id']}")
        print(f"{'='*60}")
        
        try:
            # Parse known planets if provided
            known_p = None
            if 'known_periods' in row and pd.notna(row['known_periods']):
                known_p = [float(x) for x in str(row['known_periods']).split(',')]
                known_t = [float(x) for x in str(row['known_t0s']).split(',')]
                known_d = [float(x) for x in str(row['known_durations']).split(',')]
            else:
                known_t, known_d = None, None
            
            # Process
            g_view, l_view = process_single_target(
                row['target_id'], 
                row['period'], 
                row['t0'], 
                row['duration'],
                known_p, known_t, known_d
            )
            
            global_views.append(g_view)
            local_views.append(l_view)
            labels.append(row['label'])
            
            print(f"✓ Success")
            
        except Exception as e:
            print(f"✗ Failed: {e}")
            continue
    
    # Create DataFrames
    global_df = pd.DataFrame(global_views)
    global_df['label'] = labels
    
    local_df = pd.DataFrame(local_views)
    local_df['label'] = labels
    
    # Save to CSV
    global_path = f"{output_dir}/all_global.csv"
    local_path = f"{output_dir}/all_local.csv"
    
    global_df.to_csv(global_path, index=False)
    local_df.to_csv(local_path, index=False)
    
    print(f"\n{'='*60}")
    print(f"Saved {len(global_df)} samples:")
    print(f"  Global view: {global_path} (2001 features)")
    print(f"  Local view: {local_path} (201 features)")
    print(f"{'='*60}")


# ============================================================
# EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":
    # Example 1: Single target
    print("Example 1: Processing single target")
    print("-" * 60)
    
    global_view, local_view = process_single_target(
        target_id='KIC 11442793',  # Kepler-90
        period=14.44912,
        t0=2455644.3488,
        duration=2.80,
        # Remove other planets in the system
        known_periods=[7.008, 8.719, 59.737, 91.939, 124.914, 210.60, 331.60],
        known_t0s=[2455647.8, 2455648.7, 2455698.8, 2455724.9, 
                   2455749.3, 2455839.2, 2455899.8],
        known_durations=[3.5, 4.0, 5.5, 6.0, 6.5, 7.0, 8.0]
    )
    
    print(f"\nGlobal view shape: {global_view.shape}")  # Should be (2001,)
    print(f"Local view shape: {local_view.shape}")      # Should be (201,)
    
    # Example 2: Batch processing from CSV
    print("\n\nExample 2: Batch processing")
    print("-" * 60)
    
    # Create example targets DataFrame
    targets = pd.DataFrame({
        'target_id': ['KIC 11442793', 'KIC 4852528'],
        'period': [14.44912, 14.64558],
        't0': [2455644.3488, 2455658.6073],
        'duration': [2.80, 2.12],
        'label': ['Candidate', 'Candidate']
    })
    
    # Process all targets
    # process_multiple_targets_to_csv(targets, output_dir='./processed_data')
    
    print("\nPreprocessing pipeline ready!")
    print("Modify the examples above to process your raw data.")

d:\ML_AI\heart_disease_env\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


Example 1: Processing single target
------------------------------------------------------------
Removing known planet transits...
Flattening light curve...


UnboundLocalError: cannot access local variable 'spline' where it is not associated with a value

[(10797460, 'CONFIRMED'), (10811496, 'FALSE POSITIVE'), (10848459, 'FALSE POSITIVE'), (10854555, 'CONFIRMED'), (10872983, 'CONFIRMED')]


THis the data for T

In [4]:
import pandas as pd
local_df = pd.read_csv("../datasets/all_local.csv")
global_df = pd.read_csv("../datasets/all_global.csv")

# Make datasets equal length
min_len = min(len(local_df), len(global_df))
local_df = local_df.iloc[:min_len].reset_index(drop=True)
global_df = global_df.iloc[:min_len].reset_index(drop=True)
from sklearn.preprocessing import LabelEncoder
# --- 2. Convert labels ---
encoder = LabelEncoder()
global_df["label"] = encoder.fit_transform(global_df["label"])
local_df["label"] = encoder.fit_transform(local_df["label"])

global_df.head()
local_df.head()


,0,1,2,3,4,5,6,7,8,9,...,192,193,194,195,196,197,198,199,200,label
0,1.785769,2.729798,2.548395,3.248466,1.325620,1.023493,1.503760,0.328128,1.491723,-0.609683,...,2.121168,2.081381,0.615004,2.437645,2.094423,0.313795,2.729227,1.091718,2.807821,0
1,0.587338,0.245809,-0.095720,-0.437250,-0.778779,-0.447903,-0.117028,0.213847,0.544723,0.875598,...,1.374764,1.392065,1.409367,1.409367,1.409367,1.409367,1.409367,1.409367,1.409367,0
2,0.995033,0.903242,0.941117,1.047910,1.015761,0.997172,0.965115,0.973159,0.364509,0.894915,...,1.134081,1.116810,1.125367,0.679383,1.129905,1.023179,1.082135,1.044991,1.151153,1
3,0.951343,1.128255,0.687507,1.486403,0.600641,1.163747,1.748795,1.489461,1.000000,1.079301,...,1.137279,0.926430,1.902571,1.249999,1.338689,0.468365,1.269225,1.805658,1.001325,0
4,0.139141,1.121875,3.047826,1.230143,0.240182,3.098975,0.503085,0.703715,2.584631,2.807380,...,1.014254,0.688949,2.474684,1.269588,1.697543,0.624603,0.100657,0.789521,1.478385,0


In [5]:
global_df.to_csv("../datasets/encoded_all_global.csv", index=False)
local_df.to_csv("../datasets/encoded_all_local.csv", index=False)

This function to take raw light data and normalize it and define it into gloabal and local data

### Dataset Citation

This project makes use of the processed Kepler light curves dataset made publicly available by the authors on [Mendeley Data](https://data.mendeley.com/datasets/wctcv34962/3).

Dourado Macedo, Bruno Henrique; Zalewski, Willian (2024), “Dataset_Machine_Learning_Exoplanets_2024”, Mendeley Data, V3, doi: 10.17632/wctcv34962.3


